## Backup, restore & migrate

Volumes are owned by the daemon — you can't reliably `cp -r` them, since on Docker Desktop they live inside the VM. The standard pattern is a **helper container** that mounts the volume and tars its contents.

**Back up a volume to a tarball on the host:**

```bash
docker run --rm \
  -v pgdata:/data:ro \
  -v $(pwd):/backup \
  alpine tar czf /backup/pgdata.tar.gz -C /data .
```

Mount the volume read-only at `/data`, bind-mount your current dir at `/backup`, and tar one into the other — you get `pgdata.tar.gz` in your working directory.

**Restore** is the mirror image: create a fresh volume, bind the tarball back in, extract:

```bash
docker volume create pgdata-restored
docker run --rm -v pgdata-restored:/data -v $(pwd):/backup \
  alpine tar xzf /backup/pgdata.tar.gz -C /data
```

**Migrate between hosts:** back up to a tar on host A, `scp` it to host B, restore there — the same trick moves a volume between Docker Desktop and a Linux server.

**But for a live database, don't tar the data files** — you can catch them mid-write and corrupt the backup. Use the database's own dump tool: `docker exec db pg_dump -U postgres app > app.sql`. Tar-of-the-volume is for cold data; app-level dumps are for live databases.

**Where it all lives on disk** (Linux): everything sits under `/var/lib/docker/` — `overlay2/` for image + container layers, **`volumes/<name>/_data/`** for a volume's actual contents, `containers/<id>/` for per-container runtime state and its JSON log. Two handy troubleshooting moves: `docker system df -v` breaks down what's using space, and `docker volume inspect <name>` prints the exact `_data` path so you can peek from the host.